In [11]:
import sys
import os
sys.path.append(os.path.abspath("../.."))

from app.data_providers import test_train_dataset
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import TargetEncoder
from sklearn.preprocessing import FunctionTransformer

from sklearn.calibration import CalibrationDisplay

from sklearn.pipeline import FeatureUnion
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_transformer

from sklearn.metrics import roc_auc_score, f1_score
from sklearn.metrics import classification_report

from imblearn.over_sampling import RandomOverSampler

from xgboost import XGBClassifier

import joblib

from copy import deepcopy

import shap

pd.set_option('display.max_columns', None)

In [12]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    log_loss,
    brier_score_loss
)

import tensorflow as tf
from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    BatchNormalization,
    Embedding,
    Flatten,
    Concatenate
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

In [13]:
data_train = pd.read_parquet(r"../../data/trajectory_tests/train_new_all_players_v2.parquet")
data_train.set_index('Unnamed: 0', inplace=True)
data_train = data_train.rename_axis(index=None, axis=1)

data_test = pd.read_parquet(r"../../data/trajectory_tests/test_new_all_players_v2.parquet")
data_test.set_index('Unnamed: 0', inplace=True)
data_test = data_test.rename_axis(index=None, axis=1)

In [14]:
data_train.shape

(44416, 303)

In [15]:
data_test.shape

(13112, 303)

In [16]:
data_train.head()

,shooter_slot,shooter_x,shooter_y,shooter_team_id,shot_angle,distance_to_basket_tracking,nearest_defender_dist,avg_defender_dist,defenders_within_3ft,defenders_within_5ft,defenders_within_7ft,offensive_spacing_area,ball_height,defender_closing_speed,ball_speed,ball_xy_speed,ACTION_TYPE,EVENTTIME,EVENT_TYPE,GAME_DATE,GAME_EVENT_ID,GAME_ID,GRID_TYPE,HTM,LOC_X,LOC_Y,MINUTES_REMAINING,PERIOD,PLAYER_ID,PLAYER_NAME,QUARTER,SECONDS_REMAINING,SHOT_ATTEMPTED_FLAG,SHOT_DISTANCE,SHOT_MADE_FLAG,SHOT_TIME,SHOT_TYPE,SHOT_ZONE_AREA,SHOT_ZONE_BASIC,SHOT_ZONE_RANGE,TEAM_ID,TEAM_NAME,VTM,tracking_game_clock,tracking_game_clock_old,event_GAME_ID,event_EVENTNUM,event_EVENTMSGTYPE,event_EVENTMSGACTIONTYPE,event_PERIOD,event_WCTIMESTRING,event_PCTIMESTRING,event_HOMEDESCRIPTION,event_NEUTRALDESCRIPTION,event_VISITORDESCRIPTION,event_SCORE,event_SCOREMARGIN,event_PERSON1TYPE,event_PLAYER1_ID,event_PLAYER1_NAME,event_PLAYER1_TEAM_ID,event_PLAYER1_TEAM_CITY,event_PLAYER1_TEAM_NICKNAME,event_PLAYER1_TEAM_ABBREVIATION,event_PERSON2TYPE,event_PLAYER2_ID,event_PLAYER2_NAME,event_PLAYER2_TEAM_ID,event_PLAYER2_TEAM_CITY,event_PLAYER2_TEAM_NICKNAME,event_PLAYER2_TEAM_ABBREVIATION,event_PERSON3TYPE,event_PLAYER3_ID,event_PLAYER3_NAME,event_PLAYER3_TEAM_ID,event_PLAYER3_TEAM_CITY,event_PLAYER3_TEAM_NICKNAME,event_PLAYER3_TEAM_ABBREVIATION,ball_x,ball_y,ball_z,player1_team_id,player1_id,player1_x,player1_y,player2_team_id,player2_id,player2_x,player2_y,player3_team_id,player3_id,player3_x,player3_y,player4_team_id,player4_id,player4_x,player4_y,player5_team_id,player5_id,player5_x,player5_y,player6_team_id,player6_id,player6_x,player6_y,player7_team_id,player7_id,player7_x,player7_y,player8_team_id,player8_id,player8_x,player8_y,player9_team_id,player9_id,player9_x,player9_y,player10_team_id,player10_id,player10_x,player10_y,shooter_x_t0,shooter_y_t0,defender1_dx_t0,defender1_dy_t0,defender1_dist_t0,defender2_dx_t0,defender2_dy_t0,defender2_dist_t0,defender3_dx_t0,defender3_dy_t0,defender3_dist_t0,defender4_dx_t0,defender4_dy_t0,defender4_dist_t0,defender5_dx_t0,defender5_dy_t0,defender5_dist_t0,attacker1_dx_t0,attacker1_dy_t0,attacker1_dist_t0,attacker2_dx_t0,attacker2_dy_t0,attacker2_dist_t0,attacker3_dx_t0,attacker3_dy_t0,attacker3_dist_t0,attacker4_dx_t0,attacker4_dy_t0,attacker4_dist_t0,shooter_x_t1,shooter_y_t1,defender1_dx_t1,defender1_dy_t1,defender1_dist_t1,defender2_dx_t1,defender2_dy_t1,defender2_dist_t1,defender3_dx_t1,defender3_dy_t1,defender3_dist_t1,defender4_dx_t1,defender4_dy_t1,defender4_dist_t1,defender5_dx_t1,defender5_dy_t1,defender5_dist_t1,attacker1_dx_t1,attacker1_dy_t1,attacker1_dist_t1,attacker2_dx_t1,attacker2_dy_t1,attacker2_dist_t1,attacker3_dx_t1,attacker3_dy_t1,attacker3_dist_t1,attacker4_dx_t1,attacker4_dy_t1,attacker4_dist_t1,shooter_x_t2,shooter_y_t2,defender1_dx_t2,defender1_dy_t2,defender1_dist_t2,defender2_dx_t2,defender2_dy_t2,defender2_dist_t2,defender3_dx_t2,defender3_dy_t2,defender3_dist_t2,defender4_dx_t2,defender4_dy_t2,defender4_dist_t2,defender5_dx_t2,defender5_dy_t2,defender5_dist_t2,attacker1_dx_t2,attacker1_dy_t2,attacker1_dist_t2,attacker2_dx_t2,attacker2_dy_t2,attacker2_dist_t2,attacker3_dx_t2,attacker3_dy_t2,attacker3_dist_t2,attacker4_dx_t2,attacker4_dy_t2,attacker4_dist_t2,shooter_x_t3,shooter_y_t3,defender1_dx_t3,defender1_dy_t3,defender1_dist_t3,defender2_dx_t3,defender2_dy_t3,defender2_dist_t3,defender3_dx_t3,defender3_dy_t3,defender3_dist_t3,defender4_dx_t3,defender4_dy_t3,defender4_dist_t3,defender5_dx_t3,defender5_dy_t3,defender5_dist_t3,attacker1_dx_t3,attacker1_dy_t3,attacker1_dist_t3,attacker2_dx_t3,attacker2_dy_t3,attacker2_dist_t3,attacker3_dx_t3,attacker3_dy_t3,attacker3_dist_t3,attacker4_dx_t3,attacker4_dy_t3,attacker4_dist_t3,shooter_x_t4,shooter_y_t4,defender1_dx_t4,defender1_dy_t4,defender1_dist_t4,defender2_dx_t4,defender2_dy_t4,defender2_dist_t4,defender3_dx_t4,defender3_dy_t4,defender3_dist_t4,defender4_dx_t4,defender4_dy_t4,defender4_dist_t4,defender5_dx_t4,defender5_dy_t4,defender5_dist_t

In [17]:
# ============================================================
# TARGET
# ============================================================

TARGET = "SHOT_MADE_FLAG"  


# ============================================================
# FEATURE GROUPS
# ============================================================

tracking_features = [
    # t0
    "shooter_x_t0", "shooter_y_t0",
    "defender1_dx_t0", "defender1_dy_t0", "defender1_dist_t0",
    "defender2_dx_t0", "defender2_dy_t0", "defender2_dist_t0",
    "defender3_dx_t0", "defender3_dy_t0", "defender3_dist_t0",
    "defender4_dx_t0", "defender4_dy_t0", "defender4_dist_t0",
    "defender5_dx_t0", "defender5_dy_t0", "defender5_dist_t0",
    "attacker1_dx_t0", "attacker1_dy_t0", "attacker1_dist_t0",
    "attacker2_dx_t0", "attacker2_dy_t0", "attacker2_dist_t0",
    "attacker3_dx_t0", "attacker3_dy_t0", "attacker3_dist_t0",
    "attacker4_dx_t0", "attacker4_dy_t0", "attacker4_dist_t0",

    # t1
    "shooter_x_t1", "shooter_y_t1",
    "defender1_dx_t1", "defender1_dy_t1", "defender1_dist_t1",
    "defender2_dx_t1", "defender2_dy_t1", "defender2_dist_t1",
    "defender3_dx_t1", "defender3_dy_t1", "defender3_dist_t1",
    "defender4_dx_t1", "defender4_dy_t1", "defender4_dist_t1",
    "defender5_dx_t1", "defender5_dy_t1", "defender5_dist_t1",
    "attacker1_dx_t1", "attacker1_dy_t1", "attacker1_dist_t1",
    "attacker2_dx_t1", "attacker2_dy_t1", "attacker2_dist_t1",
    "attacker3_dx_t1", "attacker3_dy_t1", "attacker3_dist_t1",
    "attacker4_dx_t1", "attacker4_dy_t1", "attacker4_dist_t1",

    # t2
    "shooter_x_t2", "shooter_y_t2",
    "defender1_dx_t2", "defender1_dy_t2", "defender1_dist_t2",
    "defender2_dx_t2", "defender2_dy_t2", "defender2_dist_t2",
    "defender3_dx_t2", "defender3_dy_t2", "defender3_dist_t2",
    "defender4_dx_t2", "defender4_dy_t2", "defender4_dist_t2",
    "defender5_dx_t2", "defender5_dy_t2", "defender5_dist_t2",
    "attacker1_dx_t2", "attacker1_dy_t2", "attacker1_dist_t2",
    "attacker2_dx_t2", "attacker2_dy_t2", "attacker2_dist_t2",
    "attacker3_dx_t2", "attacker3_dy_t2", "attacker3_dist_t2",
    "attacker4_dx_t2", "attacker4_dy_t2", "attacker4_dist_t2",

    # t3
    "shooter_x_t3", "shooter_y_t3",
    "defender1_dx_t3", "defender1_dy_t3", "defender1_dist_t3",
    "defender2_dx_t3", "defender2_dy_t3", "defender2_dist_t3",
    "defender3_dx_t3", "defender3_dy_t3", "defender3_dist_t3",
    "defender4_dx_t3", "defender4_dy_t3", "defender4_dist_t3",
    "defender5_dx_t3", "defender5_dy_t3", "defender5_dist_t3",
    "attacker1_dx_t3", "attacker1_dy_t3", "attacker1_dist_t3",
    "attacker2_dx_t3", "attacker2_dy_t3", "attacker2_dist_t3",
    "attacker3_dx_t3", "attacker3_dy_t3", "attacker3_dist_t3",
    "attacker4_dx_t3", "attacker4_dy_t3", "attacker4_dist_t3",

    # t4
    "shooter_x_t4", "shooter_y_t4",
    "defender1_dx_t4", "defender1_dy_t4", "defender1_dist_t4",
    "defender2_dx_t4", "defender2_dy_t4", "defender2_dist_t4",
    "defender3_dx_t4", "defender3_dy_t4", "defender3_dist_t4",
    "defender4_dx_t4", "defender4_dy_t4", "defender4_dist_t4",
    "defender5_dx_t4", "defender5_dy_t4", "defender5_dist_t4",
    "attacker1_dx_t4", "attacker1_dy_t4", "attacker1_dist_t4",
    "attacker2_dx_t4", "attacker2_dy_t4", "attacker2_dist_t4",
    "attacker3_dx_t4", "attacker3_dy_t4", "attacker3_dist_t4",
    "attacker4_dx_t4", "attacker4_dy_t4", "attacker4_dist_t4",

    # t5
    "shooter_x_t5", "shooter_y_t5",
    "defender1_dx_t5", "defender1_dy_t5", "defender1_dist_t5",
    "defender2_dx_t5", "defender2_dy_t5", "defender2_dist_t5",
    "defender3_dx_t5", "defender3_dy_t5", "defender3_dist_t5",
    "defender4_dx_t5", "defender4_dy_t5", "defender4_dist_t5",
    "defender5_dx_t5", "defender5_dy_t5", "defender5_dist_t5",
    "attacker1_dx_t5", "attacker1_dy_t5", "attacker1_dist_t5",
    "attacker2_dx_t5", "attacker2_dy_t5", "attacker2_dist_t5",
    "attacker3_dx_t5", "attacker3_dy_t5", "attacker3_dist_t5",
    "attacker4_dx_t5", "attacker4_dy_t5", "attacker4_dist_t5",
]

additional_features = [
    "shot_angle",
    "distance_to_basket_tracking",
    "nearest_defender_dist",
    "avg_defender_dist",
    "defenders_within_3ft",
    "defenders_within_5ft",
    "defenders_within_7ft",
    "defender_closing_speed",
    "defenders_between",
    "has_screen",
    "teammate_between_defender",
    "players_in_paint",
    "shooter_speed",
    "nearest_teammate_distance"
]

continuous_features = (
    tracking_features +
    additional_features
)

categorical_features = [
    "PERIOD"
]

player_feature = "PLAYER_ID"

In [18]:
required_columns = (
    continuous_features +
    categorical_features +
    [player_feature, TARGET]
)

data_train = data_train.dropna(subset=required_columns).copy()
data_test = data_test.dropna(subset=required_columns).copy()

In [19]:
# ============================================================
# LABEL ENCODE PLAYER IDS
# ============================================================

unique_players = data_train[player_feature].unique()

player_to_idx = {
    player_id: idx + 1
    for idx, player_id in enumerate(unique_players)
}

# 0 reserved for UNKNOWN players
UNKNOWN_PLAYER_IDX = 0


# ============================================================
# ENCODE TRAIN
# ============================================================

data_train[player_feature] = (
    data_train[player_feature]
    .map(player_to_idx)
    .fillna(UNKNOWN_PLAYER_IDX)
    .astype(int)
)


# ============================================================
# ENCODE TEST
# ============================================================

data_test[player_feature] = (
    data_test[player_feature]
    .map(player_to_idx)
    .fillna(UNKNOWN_PLAYER_IDX)
    .astype(int)
)


# ============================================================
# NUMBER OF PLAYERS
# ============================================================

num_players = max(player_to_idx.values()) + 1


# ============================================================
# TRAIN / VALIDATION / TEST SPLIT
# ============================================================

X_train = data_train.drop('SHOT_MADE_FLAG', axis=1)
y_train = data_train['SHOT_MADE_FLAG']

X_test = data_test.drop('SHOT_MADE_FLAG', axis=1)
y_test = data_test['SHOT_MADE_FLAG']

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)


In [20]:
# ============================================================
# CONTINUOUS FEATURES
# ============================================================

X_train_cont = X_train[continuous_features].values
X_valid_cont = X_val[continuous_features].values
X_test_cont = X_test[continuous_features].values


# ============================================================
# STANDARDIZATION
# ============================================================

scaler = StandardScaler()

X_train_cont = scaler.fit_transform(X_train_cont)
X_valid_cont = scaler.transform(X_valid_cont)
X_test_cont = scaler.transform(X_test_cont)


# ============================================================
# PERIOD INPUT
# ============================================================

X_train_period = X_train["PERIOD"].values
X_valid_period = X_val["PERIOD"].values
X_test_period = X_test["PERIOD"].values


# ============================================================
# PLAYER INPUT
# ============================================================

X_train_player = X_train[player_feature].values
X_valid_player = X_val[player_feature].values
X_test_player = X_test[player_feature].values


# ============================================================
# TARGET
# ============================================================

y_train = y_train.values
y_valid = y_val.values
y_test = y_test.values

In [21]:
# ============================================================
# MODEL
# ============================================================

# --------------------------------
# Continuous Input
# --------------------------------

continuous_input = Input(
    shape=(len(continuous_features),),
    name="continuous_input"
)

x = Dense(512, activation="relu")(continuous_input)
x = BatchNormalization()(x)
x = Dropout(0.30)(x)

x = Dense(256, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.25)(x)

x = Dense(128, activation="relu")(x)
x = BatchNormalization()(x)


# --------------------------------
# PLAYER EMBEDDING
# --------------------------------

player_input = Input(
    shape=(1,),
    name="player_input"
)

player_embedding = Embedding(
    input_dim=num_players + 1,
    output_dim=16,
    name="player_embedding"
)(player_input)

player_embedding = Flatten()(player_embedding)


# --------------------------------
# PERIOD EMBEDDING
# --------------------------------

period_input = Input(
    shape=(1,),
    name="period_input"
)

period_embedding = Embedding(
    input_dim=10,
    output_dim=4,
    name="period_embedding"
)(period_input)

period_embedding = Flatten()(period_embedding)


# --------------------------------
# CONCATENATE
# --------------------------------

x = Concatenate()([
    x,
    player_embedding,
    period_embedding
])

x = Dense(64, activation="relu")(x)
x = Dropout(0.20)(x)

output = Dense(
    1,
    activation="sigmoid"
)(x)


# ============================================================
# BUILD MODEL
# ============================================================

model = Model(
    inputs=[
        continuous_input,
        player_input,
        period_input
    ],
    outputs=output
)

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.AUC(name="auc")
    ]
)

model.summary()




Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ continuous_input (InputLayer) │ (None, 188)               │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 512)               │          96,768 │ continuous_input[0][0]     │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization           │ (None, 512)               │           2,048 │ dense[0][0]                │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout (Dropout)             │ (None, 512)               │               0 │ batch_normalization[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_1 (Dense)               │ (None, 256)               │         131,328 │ dropout[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_1         │ (None, 256)               │           1,024 │ dense_1[0][0]              │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_1 (Dropout)           │ (None, 256)               │               0 │ batch_normalization_1[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ player_input (InputLayer)     │ (None, 1)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ period_input (InputLayer)     │ (None, 1)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_2 (Dense)               │ (None, 128)               │          32,896 │ dropout_1[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ player_embedding (Embedding)  │ (None, 1, 16)             │           6,640 │ player_input[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ period_embedding (Embedding)  │ (None, 1, 4)              │              40 │ period_input[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_2         │ (None, 128)               │             512 │ dense_2[0][0]              │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ flatten (Flatten)             │ (None, 16)                │               0 │ player_embedding[0][0]     │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ flatten_1 (Flatten)           │ (None, 4)                 │               0 │ period_embedding[0][0]     │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ concatenate (Concatenate)     │ (None, 148)               │               

 Total params: 280,857 (1.07 MB)

 Trainable params: 279,065 (1.06 MB)

 Non-trainable params: 1,792 (7.00 KB)

In [22]:
# ============================================================
# EARLY STOPPING
# ============================================================

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)




In [23]:
# ============================================================
# TRAIN
# ============================================================

history = model.fit(
    x=[
        X_train_cont,
        X_train_player,
        X_train_period
    ],
    y=y_train,

    validation_data=(
        [
            X_valid_cont,
            X_valid_player,
            X_valid_period
        ],
        y_valid
    ),

    epochs=100,
    batch_size=256,
    callbacks=[early_stopping],
    verbose=1
)




Epoch 1/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - auc: 0.5352 - loss: 0.7162 - val_auc: 0.5594 - val_loss: 0.6886
Epoch 2/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - auc: 0.5559 - loss: 0.6908 - val_auc: 0.5743 - val_loss: 0.6846
Epoch 3/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - auc: 0.5718 - loss: 0.6830 - val_auc: 0.5728 - val_loss: 0.6859
Epoch 4/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - auc: 0.5898 - loss: 0.6767 - val_auc: 0.5793 - val_loss: 0.6818
Epoch 5/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - auc: 0.5995 - loss: 0.6730 - val_auc: 0.5727 - val_loss: 0.6835
Epoch 6/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - auc: 0.6045 - loss: 0.6713 - val_auc: 0.5771 - val_loss: 0.6843
Epoch 7/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - auc: 0.6087 - loss: 0.6695 - val_auc: 0.5761 - val_loss: 0.6828
Epoch 8/100
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - auc: 0.6139 - loss: 0.6681 - val_auc: 0.5711 - val_loss: 0.6845
Epoch 9/100
157/157 ━━━━━━━━━━━━━━━━━━━━

In [24]:
# ============================================================
# PREDICTIONS
# ============================================================

y_pred_proba = model.predict([
    X_test_cont,
    X_test_player,
    X_test_period
]).flatten()

y_pred = (y_pred_proba >= 0.5).astype(int)


# ============================================================
# EVALUATION
# ============================================================

auc = roc_auc_score(y_test, y_pred_proba)
ll = log_loss(y_test, y_pred_proba)
brier = brier_score_loss(y_test, y_pred_proba)

print("\n====================")
print(f"AUC:        {auc:.4f}")
print(f"Log Loss:   {ll:.4f}")
print(f"Brier:      {brier:.4f}")
print("====================")

410/410 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

AUC:        0.5831
Log Loss:   0.6773
Brier:      0.2422


In [26]:
display(pd.crosstab(y_test, y_pred))
print(classification_report(y_test, y_pred))

col_0,0,1
row_0,,
0,5440,1809
1,3794,2069


              precision    recall  f1-score   support

           0       0.59      0.75      0.66      7249
           1       0.53      0.35      0.42      5863

    accuracy                           0.57     13112
   macro avg       0.56      0.55      0.54     13112
weighted avg       0.56      0.57      0.55     13112



In [30]:
# Build error Dataframe

error_df = X_test.copy()

error_df["y_true"] = y_test
error_df["y_pred"] = y_pred
error_df["y_pred_proba"] = y_pred_proba

# confidence
error_df["confidence"] = np.where(
    error_df["y_pred"] == 1,
    error_df["y_pred_proba"],
    1 - error_df["y_pred_proba"]
)

error_df["correct"] = (
    error_df["y_true"] == error_df["y_pred"]
)

# prediction error magnitude
error_df["error"] = abs(error_df["y_true"] - error_df["y_pred_proba"])

In [32]:
for action_type in error_df["ACTION_TYPE"].value_counts().index[:15]:

    subset = error_df[
        error_df["ACTION_TYPE"] == action_type
    ]

    accuracy = (
        subset["y_true"] == subset["y_pred"]
    ).mean()

    print("\n" + "="*60)
    print(f"{action_type}")
    print(f"n = {len(subset)}")
    print(f"accuracy = {accuracy:.3f}")
    print(f"roc-auc = {roc_auc_score(subset["y_true"], subset["y_pred_proba"]):.3f}")
    print("="*60)

    cm = pd.crosstab(
        subset["y_true"],
        subset["y_pred"],
        rownames=["Actual"],
        colnames=["Predicted"]
    )

    display(cm)


Jump Shot
n = 6406
accuracy = 0.612
roc-auc = 0.502


Predicted,0,1
Actual,,
0,3571,797
1,1686,352



Layup Shot
n = 1286
accuracy = 0.568
roc-auc = 0.578


Predicted,0,1
Actual,,
0,460,298
1,258,270



Driving Layup Shot
n = 796
accuracy = 0.515
roc-auc = 0.553


...

Predicted,0,1
Actual,,
0,24,23
1,67,64



Driving Finger Roll Layup Shot
n = 142
accuracy = 0.542
roc-auc = 0.637


Predicted,0,1
Actual,,
0,34,14
1,51,43



Putback Layup Shot
n = 125
accuracy = 0.448
roc-auc = 0.491


Predicted,0,1
Actual,,
0,20,16
1,53,36
